In [15]:
from typing import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END


class NoticeState(TypedDict):
    draft: str
    retry_count: int
    next_step: NotRequired[str]
    final_text: NotRequired[str]


def create_draft(state: NoticeState):
    print("create_draft 실행")

    print("state : ", state)

    return {"draft": "내일 시스템 점검이 있습니다.", "retry_count": 0}


def check_draft(state: NoticeState):
    print("check_draft 실행")
    draft = state["draft"]

    print("draft : ", draft)
    print("retry_count : ", state["retry_count"])

    if len(draft) < 50:
        print("공지문이 너무 짧습니다.")
        return {"next_step": "improve_draft"}

    print("공지문 길이가 충분합니다.")

    return {"next_step": "finish_notice"}


def improve_draft(state: NoticeState):
    print("improve_draft 실행")

    draft = state["draft"]
    retry_count = state["retry_count"]

    improved_draft = (
        draft
        + "오전 2시부터 4시까지 시스템 점검이 진행됩니다."
        + "점검 시간동안 서비스 이용이 일시적으로 제한될 수 있습니다."
    )

    return {"draft": improved_draft, "retry_count": retry_count + 1}


def finish_notice(state: NoticeState):
    print("finish 실행")

    final_text = state["draft"]

    print("최종 공지문 : ", final_text)

    return {"final_text": final_text}


def route_after_check(state: NoticeState):
    if state["next_step"] == "finish_notice":
        return "finish_notice"

    if state["retry_count"] >= 2:
        return "finish_notice"

    return "improve_draft"


graph_builder = StateGraph(NoticeState)

graph_builder.add_node("create_draft", create_draft)
graph_builder.add_node("check_draft", check_draft)
graph_builder.add_node("improve_draft", improve_draft)
graph_builder.add_node("finish_notice", finish_notice)

graph_builder.add_edge(START, "create_draft")
graph_builder.add_edge("create_draft", "check_draft")
graph_builder.add_edge("improve_draft", "check_draft")

graph_builder.add_conditional_edges(
    "check_draft",
    route_after_check,
    {"improve_draft": "improve_draft", "finish_notice": "finish_notice"},
)

graph_builder.add_edge("finish_notice", END)

graph = graph_builder.compile()

result = graph.invoke({"draft": "", "retry_count": 0})

create_draft 실행
state :  {'draft': '', 'retry_count': 0}
check_draft 실행
draft :  내일 시스템 점검이 있습니다.
retry_count :  0
공지문이 너무 짧습니다.
improve_draft 실행
check_draft 실행
draft :  내일 시스템 점검이 있습니다.오전 2시부터 4시까지 시스템 점검이 진행됩니다.점검 시간동안 서비스 이용이 일시적으로 제한될 수 있습니다.
retry_count :  1
공지문 길이가 충분합니다.
finish 실행
최종 공지문 :  내일 시스템 점검이 있습니다.오전 2시부터 4시까지 시스템 점검이 진행됩니다.점검 시간동안 서비스 이용이 일시적으로 제한될 수 있습니다.


In [18]:
import os
from dotenv import load_dotenv
from typing import TypedDict, NotRequired
from langchain_core.documents import Document
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END


load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING = os.getenv("NVIDIA_EMBEDDING")

llm = ChatNVIDIA(api_key=API_KEY, model=MODEL)

embeddings = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING)


class RagState(TypedDict):
    question: str
    search_query: str
    documents: list
    retry_count: int
    next_step: NotRequired[str]
    answer: NotRequired[str]


documents = [
    Document(
        page_content=(
            "ISA는 개인종합자산관리계좌로 여러 금융상품을 하나의 계좌에서 "
            "통합하여 관리할 수 있습니다."
        )
    ),
    Document(
        page_content=(
            "ISA를 활용하면 일정한 요건을 충족하는 경우 금융상품에서 발생한 "
            "소득에 대해 세제 혜택을 받을 수 있습니다."
        )
    ),
    Document(
        page_content=(
            "ISA에는 가입 대상과 납입 한도 등의 조건이 있으며, "
            "상품 유형에 따라 운용 방법과 세부 조건이 달라질 수 있습니다."
        )
    ),
    Document(
        page_content=(
            "주식 투자에서는 투자 원금의 손실이 발생할 수 있으며 "
            "투자자는 상품의 위험성을 충분히 확인해야 합니다."
        )
    ),
]


vector_store = FAISS.from_documents(documents=documents, embedding=embeddings)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})


def retriever_documents(state: RagState):
    print("retriever_documents")

    query = state["search_query"]

    print("query : ", query)

    docs = retriever.invoke(query)

    print(len(docs))

    return {"documents": docs}


def grade_documents(state: RagState):
    print("grade_documents")

    docs = state["documents"]

    for doc in docs:
        print("-", doc.page_content)

    if len(docs) >= 2:
        return {"next_step": "generate"}

    return {"next_step": "rewrite"}


def rewrite_query(state: RagState):
    print("rewrite_query")

    question = state["question"]
    retry_count = state["retry_count"]

    new_query = question + "세제 혜택 가입 조건"

    print("기존 질문  : ", question)
    print("새 검색어 : ", new_query)

    return {"search_query": new_query, "retry_count": retry_count + 1}


def generate_answer(state: RagState):
    print("generate_answer")

    docs = state["documents"]

    context = "\n".join(doc.page_content for doc in docs)

    answer = f"""
            질문:
            {state["question"]}

            참고 문서:
            {context}

            답변:
            제공된 문서에 따르면 ISA는 여러 금융상품을 하나의 계좌에서 관리할 수 있으며,
            일정 요건을 충족하면 세제 혜택을 받을 수 있습니다.
            """

    print(answer)

    return {"answer": answer}


def route_after_grade(state: RagState):
    if state["next_step"] == "generate":
        return "generate"

    if state["retry_count"] >= 2:
        return "generate"

    return "rewrite"


graph_builder = StateGraph(RagState)

graph_builder.add_node("retriever", retriever_documents)
graph_builder.add_node("grade", grade_documents)
graph_builder.add_node("rewrite", rewrite_query)
graph_builder.add_node("generate", generate_answer)

graph_builder.add_edge(START, "retriever")
graph_builder.add_edge("retriever", "grade")
graph_builder.add_edge("rewrite", "retriever")

graph_builder.add_conditional_edges(
    "grade", route_after_grade, {"rewrite": "rewrite", "generate": "generate"}
)

graph = graph_builder.compile()

result = graph.invoke(
    {"question": "ISA의 장점은 무엇인가?", "search_query": "ISA 장점", "retry_count": 0}
)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


retriever_documents
query :  ISA 장점
2
grade_documents
- ISA를 활용하면 일정한 요건을 충족하는 경우 금융상품에서 발생한 소득에 대해 세제 혜택을 받을 수 있습니다.
- ISA는 개인종합자산관리계좌로 여러 금융상품을 하나의 계좌에서 통합하여 관리할 수 있습니다.
generate_answer

            질문:
            ISA의 장점은 무엇인가?

            참고 문서:
            ISA를 활용하면 일정한 요건을 충족하는 경우 금융상품에서 발생한 소득에 대해 세제 혜택을 받을 수 있습니다.
ISA는 개인종합자산관리계좌로 여러 금융상품을 하나의 계좌에서 통합하여 관리할 수 있습니다.

            답변:
            제공된 문서에 따르면 ISA는 여러 금융상품을 하나의 계좌에서 관리할 수 있으며,
            일정 요건을 충족하면 세제 혜택을 받을 수 있습니다.
            


In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"


llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_tokens=4096)

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=API_KEY,
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents=documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

retriever = vector_store.as_retriever(search_kwargs={"k": 5})


class RAGState(TypedDict):
    question: str
    search_query: str
    documents: list
    retry_count: int
    relevant: bool
    answer: str


class GradeDocuments(BaseModel):
    relevant: bool = Field(
        description="검색된 문서가 사용자의 질문에 답변하는데 충분히 관련되어 있으면 True, 그렇지 않으면 False"
    )


grader = llm.with_structured_output(GradeDocuments)


def retrieve_documents(state: RAGState):
    print("retrieve_documents")

    query = state["search_query"]

    print("query : ", query)

    documents = retriever.invoke(query)

    print("검색된 문서 : ", len(documents))

    return {"documents": documents}


def grade_documents(state: RAGState):
    print("grade_documents")

    question = state["question"]
    documents = state["documents"]

    context = "\n\n".join(doc.page_content for doc in documents)

    prompt = f"""
            당신은 RAG 검색 결과 평가자입니다.

            사용자의 질문:
            {question}

            검색된 문서:
            {context}

            판단 기준:

            1. 검색된 문서가 질문의 핵심 내용과 관련되어 있는가?
            2. 이 문서를 이용해서 질문에 답변할 수 있는가?

            답변에 필요한 정보가 충분히 포함되어 있다면
            relevant=True

            관련성이 낮거나 질문에 답변하기 어렵다면
            relevant=False
            """
    result = grader.invoke(prompt)

    print("문서 관련성 : ", result.relevant)

    return {"relevant": result.relevant}


def rewrite_query(state: RAGState):
    print("rewrite_query")

    question = state["question"]

    prompt = f"""
            사용자의 질문을 검색 시스템에서 더 좋은 결과를 얻을 수 있도록
            검색어로 다시 작성하세요.

            사용자 질문:
            {question}

            검색에 중요한 핵심 키워드를 포함하세요.

            검색어만 출력하세요.
            """

    response = llm.invoke(prompt)

    new_query = response.content.strip()

    retry_count = state["retry_count"] + 1

    print("기존 검색어:", state["search_query"])
    print("새 검색어:", new_query)
    print("재검색 횟수:", retry_count)

    return {
        "search_query": new_query,
        "retry_count": retry_count,
    }


def generate_answer(state: RAGState):
    print("generate_answer")

    question = state["question"]
    documents = state["documents"]

    context = "\n\n".join(doc.page_content for doc in documents)

    prompt = f"""
            다음 문서를 기반으로 사용자의 질문에 답변하세요.

            [문서]
            {context}

            [질문]
            {question}

            규칙:

            - 제공된 문서에 근거해서 답변하세요.
            - 문서에서 확인할 수 없는 내용은 추측하지 마세요.
            - 가능하면 핵심 내용을 이해하기 쉽게 설명하세요.
            """

    response = llm.invoke(prompt)

    return {"answer": response}


def route_after_grade(state: RAGState):
    if state["relevant"]:
        return "generate"

    if state["retry_count"] >= 2:
        return "generate"

    return "rewrite"


builder = StateGraph(RAGState)

builder.add_node(
    "retrieve",
    retrieve_documents,
)

builder.add_node(
    "grade",
    grade_documents,
)

builder.add_node(
    "rewrite",
    rewrite_query,
)

builder.add_node(
    "generate",
    generate_answer,
)


builder.add_edge(
    START,
    "retrieve",
)

builder.add_edge(
    "retrieve",
    "grade",
)


builder.add_conditional_edges(
    "grade",
    route_after_grade,
    {
        "rewrite": "rewrite",
        "generate": "generate",
    },
)


builder.add_edge(
    "rewrite",
    "retrieve",
)


builder.add_edge(
    "generate",
    END,
)


graph = builder.compile()

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

initial_state = {
    "question": question,
    "search_query": question,
    "documents": [],
    "retry_count": 0,
    "relevant": False,
    "answer": "",
}


result = graph.invoke(initial_state)

print("최종 결과")
print("=" * 60)

print(result["answer"])

/var/folders/cn/n1wp5rkx27j7415_2rb44sk00000gn/T/ipykernel_19596/618252828.py:19: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600, max_tokens=4096)
/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


retrieve_documents
query :  모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?
검색된 문서 :  5
grade_documents
